# 인공지능 플랫폼실습 개인 과제 — 웨어체크

| 항목 | 내용 |
|---|---|
| 작성자 | 박수용 |
| 학번 | 2601110288 |
| 제출일 | 2026년 10월 6일 |
| 주제 | 상품 사진과 등록 카테고리 비교 |
| 데모 주소 | https://huggingface.co/spaces/psy0635/fashion-image-check |

이 파일에는 제작 내용, 실행 방법 및 전체 소스를 정리하였습니다. 코드 셀을 실행하면 `index.html`이 생성됩니다. 생성된 파일의 HTML·CSS·JavaScript가 화면 구성과 AI 분석을 수행하며, 노트북의 Python 커널에서는 AI 모델을 실행하지 않습니다.


## 1. 제작 목적과 입력·출력

상품을 등록할 때 사진과 카테고리가 서로 다르게 입력될 수 있습니다. 웨어체크는 상품 사진을 분석한 결과와 사용자가 선택한 카테고리를 비교하여 다시 확인할 항목을 안내하는 데모입니다. 비교 범위는 상의, 하의, 신발 세 가지로 정하였습니다.

| 구분 | 내용 |
|---|---|
| 입력 | 상품 사진과 등록 카테고리(상의·하의·신발) |
| 선택 입력 | 상품 이름. 기록용으로 사용하며 AI 분석에는 사용하지 않습니다. |
| 출력 | 등록값과 추천의 일치, 확인 필요, 판단 보류 |
| 판정 근거 | 후보별 비교 점수, AI 추천, 분류 시간 |
| 부가 기능 | 판단 기준 조절, 결과 복사, 결과 TXT 저장 |

사진은 JPG, PNG, WebP 형식을 지원하며 파일 크기는 10MB 이하로 제한하였습니다. 파일의 확장자만 바꾸는 것으로는 사진 형식이 변환되지 않습니다.

## 2. 사용 모델과 선택 이유

| 항목 | 내용 |
|---|---|
| 모델 | `Xenova/clip-vit-base-patch32` |
| 작업 | `zero-shot-image-classification` |
| 실행 도구 | Transformers.js 3.7.6 |
| 실행 설정 | WebAssembly(`wasm`), 8비트 양자화(`q8`) |
| 실행 장소 | 사용자의 웹 브라우저 |
| 배포 방식 | Hugging Face Static Space |

CLIP은 사진과 글로 된 후보 설명을 비교하는 모델입니다. 카테고리 후보를 코드에서 지정할 수 있고 별도 추가 학습 없이 비교할 수 있어 선택하였습니다. 제로샷 분류는 이 과제의 사진으로 모델을 추가 학습시키지 않고 후보 설명을 이용하여 분류하는 방식입니다.

| 카테고리 | 모델에 전달하는 후보 설명 |
|---|---|
| 상의 | `a photo of a shirt or a top` |
| 하의 | `a photo of pants or a skirt` |
| 신발 | `a photo of shoes` |

처음 실행할 때는 인터넷을 통해 모델 파일과 실행 도구를 내려받습니다. 이 코드에는 상품 사진을 서버로 업로드하는 요청을 작성하지 않았으며, 분석은 브라우저에서 수행합니다.


## 3. 처리 과정과 판정 기준

1. 등록 카테고리와 상품 사진을 선택합니다.
2. 파일 형식·크기를 확인하고 미리보기를 표시합니다.
3. CLIP으로 사진과 세 후보 설명을 비교합니다.
4. 후보별 점수를 내림차순으로 정렬합니다.
5. 1위 점수와 1·2위 차이가 기준을 충족하는지 확인합니다.
6. 기준을 충족하면 추천과 등록값을 비교하고, 충족하지 못하면 판단을 보류합니다.

| 조건 | 결과 |
|---|---|
| 1위 점수 65% 미만 **또는** 1·2위 차이 15%p 미만 | 판단 보류 |
| 두 기준을 충족하고 추천과 등록값이 같음 | 등록값과 추천이 같습니다 |
| 두 기준을 충족하고 추천과 등록값이 다름 | 사진 또는 카테고리 확인 필요 |

1·2위 차이는 `(1위 점수 − 2위 점수) × 100`으로 계산합니다. %p는 두 백분율의 차이를 나타내는 단위입니다. 기준과 정확히 같은 값은 충족으로 처리합니다. 65%와 15%p는 시연용 설정이며 실제 상품 데이터로 검증한 기준은 아닙니다.

기존 PPT의 상의 사례에서는 1위 점수가 94.71%, 1·2위 차이가 90.15%p였습니다. 두 기준을 충족하고 등록값과 추천이 모두 상의이므로 일치 결과에 해당합니다. 이 수치는 기존 PPT의 사례를 정리한 것이며, 본 노트북에서 새로 측정한 값은 아닙니다.

후보별 점수는 세 후보 사이의 상대 비교 값입니다. **모델의 정확도나 실제 정답 확률로 해석할 수 없습니다.** 화면은 점수를 소수점 한 자리까지 표시하지만 판정에는 반올림 전 값을 사용합니다.

## 4. 구현 중 확인한 문제와 해결 방법

결과만 표시하면 사용자가 판정 이유를 알기 어려웠습니다. 후보별 점수를 표시하고 판단 기준을 조절할 수 있도록 구성하였습니다. PPT에서는 1위 점수와 후보 간 차이의 기준 충족 여부를 표로 설명하였습니다.

같은 사진에서 등록 카테고리를 바꾸면 기존 분석 결과를 재사용하여 다시 비교하도록 하였습니다. 판단 기준을 바꿀 때에도 기존 점수로 판정만 다시 계산합니다. 사진을 새로 선택하면 이전 결과를 지워 서로 다른 사진의 결과가 섞이지 않도록 하였습니다.

## 5. 한계와 개선 방향

1. **세 카테고리만 비교합니다.** 가방이나 액세서리 등 후보 밖의 물건도 세 후보 중 하나로 추천될 수 있습니다.
2. **사진에 따라 잘못 분류할 수 있습니다.** 여러 상품이나 복잡한 배경에서는 높은 점수로 잘못 추천할 가능성도 있습니다.
3. **판정 기준을 실제 상품 데이터로 검증하지 않았습니다.** 현재 결과만으로 전체 정확도를 평가할 수 없습니다.

실제 서비스에 적용하려면 정답 카테고리가 있는 상품 사진으로 성능과 판정 기준을 검증해야 합니다. 여러 사진의 일괄 처리, 검수 이력 저장, 사용자 최종 확인 기능도 필요합니다. 색상·브랜드·정품 여부·소재는 현재 검사 범위에 포함하지 않았습니다.


## 6. 핵심 코드 설명

| 코드 | 설명 | 역할 |
|---|---|---|
| `categories` | 카테고리의 ID·이름·영문 설명을 저장합니다. | 분류 후보 정의 |
| `pipeline('zero-shot-image-classification', ...)` | 모델과 브라우저 실행 설정을 불러옵니다. | 모델 준비 |
| `classifier(imageUrl, categories.map(...))` | 사진과 후보 설명을 모델에 전달합니다. | 점수 계산 |
| `evaluate()` | 점수·차이 기준과 등록값을 비교합니다. | 판정·화면 표시 |
| `sync()` | 모델·사진·처리 상태에 따라 버튼을 활성화합니다. | 입력 흐름 관리 |
| `clearResult()` | 이전 판정과 후보 점수를 지웁니다. | 결과 초기화 |
| `report()` | 점검 결과를 텍스트로 구성합니다. | 복사·TXT 저장 |

전체 실행 소스는 아래 코드 셀에 포함하였습니다.

## 7. 실행 방법

**배포된 데모 실행**

1. 위 데모 주소를 Chrome 또는 Edge에서 엽니다.
2. 상태 안내에 ‘준비 완료’가 표시될 때까지 기다립니다.
3. 등록 카테고리와 사진을 선택하고 점검 버튼을 누릅니다.
4. 후보 점수와 판정 결과를 확인합니다.

**노트북에서 소스 생성**

1. 이 파일을 Google Colab 또는 Jupyter에서 엽니다.
2. 아래 ‘전체 소스’ 코드 셀을 실행하여 `index.html`을 생성합니다. 같은 작업 폴더에 `index.html`이 있으면 덮어씁니다.
3. Colab에서는 왼쪽 파일 목록을 새로고침하고 `index.html`을 내려받습니다. Jupyter에서는 현재 작업 폴더에서 파일을 찾습니다.
4. 파일을 Chrome 또는 Edge에서 엽니다. 모델 파일을 받을 수 있도록 인터넷 연결이 필요합니다.
5. 파일을 직접 열어 실행되지 않으면 배포 주소를 이용하거나, PC에서 HTML이 있는 폴더의 터미널에 `python -m http.server 8000`을 입력하고 `http://localhost:8000`을 엽니다. 서버 종료는 터미널에서 `Ctrl+C`를 누릅니다.

HTML 생성에는 모델 설치가 필요하지 않습니다. 노트북 내부 미리보기에는 브라우저 권한 제한이 있을 수 있으므로 생성된 파일이나 배포 주소에서 실행합니다.

## 8. 전체 소스 — index.html 생성

다음 셀은 제출한 `index(3).html`의 전체 소스를 포함합니다. 전체 HTML을 문자열로 보관하고 Python의 `Path.write_text()`로 UTF-8 파일을 생성합니다. HTML의 JavaScript는 생성된 파일을 브라우저에서 열 때 실행됩니다.


In [1]:
from pathlib import Path

# 아래 HTML·CSS·JavaScript 전체를 UTF-8 파일로 저장합니다.
html_source = r"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>웨어체크 · AI 의류 등록 점검</title>
<style>
:root{--ink:#182a28;--muted:#667571;--green:#216a54;--line:#dfe6df;--bg:#f5f6f0}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:"맑은 고딕",system-ui,sans-serif}header,main{max-width:1120px;margin:auto;padding:24px}header{display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line)}.brand{font-weight:900;font-size:22px}.tag,.pill{border-radius:30px;padding:7px 12px;font-size:12px;background:#e6eee7}.hero{margin:20px 0 26px}.eyebrow{font-size:12px;font-weight:800;letter-spacing:2px;color:var(--green)}h1{font-size:36px;letter-spacing:-1.5px;margin:12px 0}p{line-height:1.7}.muted{color:var(--muted)}.grid{display:grid;grid-template-columns:1fr 1fr;gap:22px}.card{background:white;border:1px solid var(--line);border-radius:20px;padding:26px}h2{font-size:19px;margin:0 0 20px}.step{color:var(--green);margin-right:9px}label{display:block;font-size:13px;font-weight:700;margin:18px 0 9px}select,input[type=text]{width:100%;padding:12px;border:1px solid #cad6cd;border-radius:9px;font:inherit;background:white}.upload{border:2px dashed #cbd9cc;background:#f7faf6;border-radius:14px;padding:24px;text-align:center;cursor:pointer}.upload strong{display:block;margin-bottom:8px}.upload small{color:var(--muted)}#upload{max-width:100%;margin-top:16px}#previewWrap{height:270px;background:#f5f7f2;border-radius:12px;margin-top:16px;display:flex;align-items:center;justify-content:center;overflow:hidden}#preview{max-width:100%;max-height:100%;object-fit:contain;display:none}button{font:inherit;cursor:pointer;border:0;border-radius:10px;padding:12px 16px}button:disabled{opacity:.5;cursor:not-allowed}.primary{background:var(--green);color:white;font-weight:700;width:100%;margin-top:18px}.secondary{background:#eef2ec;color:var(--ink);font-size:13px}#status{padding:12px;border-radius:9px;background:#eef2ed;font-size:13px;margin:18px 0}#verdict{background:#f3f5f0;border-radius:14px;padding:22px;min-height:160px}#verdict[data-kind=match]{background:#e6f3e9}#verdict[data-kind=mismatch]{background:#fff0de}#verdict[data-kind=hold]{background:#eef0f6}#verdict h3{margin:0 0 10px;font-size:23px}#verdict p{margin:0;font-size:14px}.summary{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:20px 0}.metric{background:#f7f8f4;padding:13px;border-radius:10px}.metric span{display:block;font-size:11px;color:var(--muted)}.metric strong{display:block;margin-top:8px;font-size:17px}.score-row{margin:16px 0}.score-title{display:flex;justify-content:space-between;font-size:13px;margin-bottom:7px}.track{height:9px;background:#edf1ea;border-radius:8px;overflow:hidden}.bar{height:100%;background:#7aa58b;width:0}.score-row:first-child .bar{background:var(--green)}.actions{display:flex;gap:8px;margin-top:20px;flex-wrap:wrap}.hint{font-size:12px;line-height:1.7;color:var(--muted)}details{margin-top:22px;border-top:1px solid var(--line);padding-top:18px}summary{cursor:pointer;font-size:13px;font-weight:700}input[type=range]{width:100%;accent-color:var(--green)}.bottom{margin-top:22px;display:grid;grid-template-columns:1fr 1fr;gap:22px}.bottom h2{margin-bottom:12px}.bottom p{font-size:13px}.notice{font-size:12px;color:var(--muted);margin-top:24px}#fileInfo{font-size:12px;overflow-wrap:anywhere}#actionNotice{min-height:20px;font-size:12px;color:var(--green)}@media(max-width:760px){header,main{padding:18px}.grid,.bottom{grid-template-columns:1fr}h1{font-size:28px}.card{padding:20px}.summary{gap:5px}.metric{padding:10px}.metric strong{font-size:15px}}
</style>
</head>
<body>
<header><div class="brand">웨어체크 <span style="color:#799181">/</span></div><span class="tag">AI 의류 등록 점검 · 발표 데모</span></header>
<main>
<section class="hero"><div class="eyebrow">FASHION CATALOG CHECK</div><h1>등록한 종류와 상품 사진,<br>같은 옷을 가리키고 있나요?</h1><p class="muted">상품 사진을 분석하고 등록 카테고리와 비교해, 다시 확인할 항목을 알려 줍니다.</p></section>
<div class="grid">
<section class="card"><h2><span class="step">01</span>상품 정보와 사진</h2>
<label for="product">상품 이름 <span class="muted">· 선택 사항</span></label><input id="product" type="text" maxlength="100" placeholder="예: 블랙 와이드 팬츠">
<p class="hint">상품 이름은 기록용이며 AI 분석에 사용하지 않습니다.</p>
<label for="category">등록 카테고리</label><select id="category"><option value="">카테고리를 선택하세요</option><option value="top">상의</option><option value="bottom">하의</option><option value="shoes">신발</option></select>
<label for="upload">상품 사진</label><div class="upload"><strong>상품 한 개가 잘 보이는 사진을 선택하세요</strong><small>JPG · PNG · WebP / 최대 10MB</small><input id="upload" type="file" accept="image/jpeg,image/png,image/webp" disabled></div>
<div id="previewWrap"><span id="placeholder" class="muted">선택한 사진이 여기에 표시됩니다</span><img id="preview" alt="선택한 상품 사진"></div><p id="fileInfo" class="muted">사진을 선택하지 않았습니다.</p>
<button id="run" class="primary" disabled>사진 분석하고 등록값 점검하기</button><button id="reset" class="secondary" style="margin-top:10px">입력 초기화</button>
</section>
<section class="card"><h2><span class="step">02</span>등록 점검 결과</h2><div id="status" role="status" aria-live="polite">AI 모델을 불러오는 중입니다. 첫 실행은 시간이 걸릴 수 있습니다.</div>
<div id="verdict" aria-live="polite"><h3>분석을 기다리고 있습니다</h3><p>카테고리와 사진을 선택한 뒤 점검 버튼을 누르세요.</p></div>
<div class="summary"><div class="metric"><span>등록 카테고리</span><strong id="registered">—</strong></div><div class="metric"><span>AI 추천</span><strong id="recommended">—</strong></div><div class="metric"><span>분류 시간</span><strong id="elapsed">—</strong></div></div>
<h2 style="font-size:15px">후보별 비교 점수</h2><div id="scores"><p class="hint">사진을 분석하면 세 후보의 점수를 표시합니다.</p></div><p class="hint">점수는 상의·하의·신발 후보 간 상대 비교 값입니다. 정확도나 실제 정답 확률을 뜻하지 않습니다.</p>
<details><summary>판단 보류 기준 조절</summary><label for="threshold">1위 최소 점수: <output id="thresholdValue">65</output>%</label><input id="threshold" type="range" min="40" max="90" value="65" step="5"><label for="margin">1·2위 최소 차이: <output id="marginValue">15</output>%p</label><input id="margin" type="range" min="5" max="40" value="15" step="5"><p class="hint">둘 중 하나라도 기준에 못 미치면 판단을 보류합니다. 기본값은 시연용 설정이며 실제 데이터로 검증한 기준이 아닙니다. 변경하면 기존 결과를 다시 점검합니다.</p></details>
<div class="actions"><button id="copy" class="secondary" disabled>점검 결과 복사</button><button id="download" class="secondary" disabled>결과 TXT 저장</button></div><p id="actionNotice" role="status"></p>
</section></div>
<div class="bottom"><section class="card"><h2>발표에서 이렇게 비교하세요</h2><p><b>① 일치 사례</b> · 바지 사진 + 하의 선택<br><b>② 불일치 사례</b> · 같은 사진 + 신발 선택<br><b>③ 판단 보류</b> · 기준을 조절해 안내 변화 확인</p><p class="hint">③은 판단 기준 기능의 시연입니다. 모델 자체의 오분류 사례는 별도 사진으로 확인하세요.</p></section><section class="card"><h2>사람의 최종 확인이 필요합니다</h2><p>전체 코디, 여러 상품, 후보 밖의 물건은 잘못 추천할 수 있습니다. 점수가 높아도 오류 가능성이 있으며, 일치 결과가 등록 정보 전체의 정확성을 보장하지 않습니다.</p><p class="hint">범위: 상품 한 개의 상의·하의·신발 비교. 색상·브랜드·정품·소재는 검사하지 않습니다.</p></section></div>
<p class="notice">모델: Xenova/clip-vit-base-patch32 · 상품 사진은 이 코드에서 서버로 업로드하지 않고 브라우저 안에서 분석합니다. 모델 파일을 받으려면 인터넷 연결이 필요합니다.</p>
</main>
<script type="module">
const $=id=>document.getElementById(id);
const categories=[{id:'top',name:'상의',description:'a photo of a shirt or a top'},{id:'bottom',name:'하의',description:'a photo of pants or a skirt'},{id:'shoes',name:'신발',description:'a photo of shoes'}];
let classifier=null,imageUrl=null,busy=false,lastResult=null;
function sync(){ $('run').disabled=!classifier||!imageUrl||!$('category').value||busy;for(const id of ['upload','category','product','reset'])$(id).disabled=busy||(id==='upload'&&!classifier);for(const id of ['copy','download'])$(id).disabled=!lastResult||busy;}
function clearResult(){lastResult=null;$('verdict').removeAttribute('data-kind');$('verdict').replaceChildren();const h=document.createElement('h3');h.textContent='분석을 기다리고 있습니다';const p=document.createElement('p');p.textContent='카테고리와 사진을 선택한 뒤 점검 버튼을 누르세요.';$('verdict').append(h,p);for(const id of ['registered','recommended','elapsed'])$(id).textContent='—';$('scores').textContent='사진을 분석하면 세 후보의 점수를 표시합니다.';$('actionNotice').textContent='';sync();}
function evaluate(){if(!lastResult)return;const r=lastResult.results;const top=r[0],gap=(top.score-r[1].score)*100;const minimum=Number($('threshold').value),difference=Number($('margin').value);const selected=$('category').value;const registered=categories.find(c=>c.id===selected);const hold=top.score*100<minimum||gap<difference;const kind=hold?'hold':top.id===selected?'match':'mismatch';const title={hold:'판단 보류',match:'등록값과 추천이 같습니다',mismatch:'사진 또는 카테고리 확인 필요'}[kind];const message=hold?`1위 점수 또는 후보 간 차이가 설정 기준에 못 미칩니다. 상품이 잘 보이는 사진으로 확인하세요.`:kind==='match'?`등록 카테고리와 AI 추천이 모두 ${top.name}입니다. 실제 상품을 마지막으로 확인하세요.`:`등록은 ${registered.name}, AI 추천은 ${top.name}입니다. 사진을 잘못 올렸거나 카테고리를 잘못 선택했는지 확인하세요.`;
lastResult.verdict={kind,title,message,registered:registered.name,minimum,difference,gap};$('verdict').dataset.kind=kind;$('verdict').replaceChildren();const h=document.createElement('h3');h.textContent=title;const p=document.createElement('p');p.textContent=message;$('verdict').append(h,p);$('registered').textContent=registered.name;$('recommended').textContent=top.name;$('elapsed').textContent=lastResult.seconds+'초';$('scores').replaceChildren();for(const item of r){const row=document.createElement('div');row.className='score-row';const title=document.createElement('div');title.className='score-title';const name=document.createElement('span');name.textContent=item.name;const score=document.createElement('b');score.textContent=(item.score*100).toFixed(1)+'%';title.append(name,score);const track=document.createElement('div');track.className='track';const bar=document.createElement('div');bar.className='bar';bar.style.width=(item.score*100)+'%';track.append(bar);row.append(title,track);$('scores').append(row);}sync();}
$('upload').addEventListener('change',async()=>{clearResult();if(imageUrl)URL.revokeObjectURL(imageUrl);imageUrl=null;$('preview').removeAttribute('src');$('preview').style.display='none';$('placeholder').hidden=false;const file=$('upload').files[0];$('fileInfo').textContent='사진을 선택하지 않았습니다.';sync();if(!file)return;if(!['image/jpeg','image/png','image/webp'].includes(file.type)){$('status').textContent='JPG, PNG, WebP 형식으로 실제 변환한 사진을 선택하세요.';return;}if(file.size>10*1024*1024){$('status').textContent='10MB 이하의 사진을 선택하세요.';return;}busy=true;sync();const candidate=URL.createObjectURL(file);try{$('preview').src=candidate;await $('preview').decode();imageUrl=candidate;$('preview').style.display='block';$('placeholder').hidden=true;$('fileInfo').textContent=`${file.name} · ${(file.size/1024/1024).toFixed(2)}MB · ${$('preview').naturalWidth} × ${$('preview').naturalHeight}`;$('status').textContent='사진 준비 완료. 등록 카테고리를 선택하고 점검하세요.';}catch(e){URL.revokeObjectURL(candidate);$('preview').removeAttribute('src');$('status').textContent='사진을 읽지 못했습니다. 실제 JPG 또는 PNG로 변환해 주세요.';}finally{busy=false;sync();}});
$('category').addEventListener('change',()=>{if(lastResult&&$('category').value){evaluate();$('status').textContent='기존 사진 분석 결과로 등록값을 다시 비교했습니다.';}else if(lastResult)clearResult();sync();});
for(const id of ['threshold','margin'])$(id).addEventListener('input',()=>{$(id+'Value').textContent=$(id).value;evaluate();});
$('reset').addEventListener('click',()=>{if(imageUrl)URL.revokeObjectURL(imageUrl);imageUrl=null;$('upload').value='';$('category').value='';$('product').value='';$('preview').removeAttribute('src');$('preview').style.display='none';$('placeholder').hidden=false;$('fileInfo').textContent='사진을 선택하지 않았습니다.';clearResult();$('status').textContent=classifier?'준비 완료. 새 상품을 입력하세요.':'모델을 불러오는 중입니다.';});
$('run').addEventListener('click',async()=>{if(!classifier||!imageUrl||!$('category').value||busy)return;busy=true;clearResult();sync();$('status').textContent='사진을 분석하고 있습니다...';try{const start=performance.now();const output=await classifier(imageUrl,categories.map(c=>c.description));const results=output.map(item=>{const c=categories.find(c=>c.description===item.label);if(!c||!Number.isFinite(item.score))throw Error('분류 결과 형식을 확인할 수 없습니다.');return {...c,score:item.score};}).sort((a,b)=>b.score-a.score);if(results.length!==3)throw Error('세 후보의 결과를 받지 못했습니다.');lastResult={results,seconds:((performance.now()-start)/1000).toFixed(1),file:$('upload').files[0].name,date:new Date().toLocaleString('ko-KR')};evaluate();$('status').textContent='점검 완료. 등록 카테고리를 바꾸면 다시 비교할 수 있습니다.';}catch(e){$('status').textContent='분석 실패: '+e.message;console.error(e);}finally{busy=false;sync();}});
function report(){const r=lastResult,v=r.verdict;return ['웨어체크 · AI 의류 등록 점검',`기록 시각: ${r.date}`,`상품 이름: ${$('product').value.trim()||'미입력'}`,`사진: ${r.file}`,`등록 카테고리: ${v.registered}`,`AI 추천: ${r.results[0].name}`,`결과: ${v.title}`,v.message,`분류 시간: ${r.seconds}초`,`판단 기준: 1위 ${v.minimum}% 이상 / 1·2위 차이 ${v.difference}%p 이상`,'',...r.results.map(c=>`${c.name}: ${(c.score*100).toFixed(1)}%`),'','점수는 후보 간 상대 비교 값이며 정확도나 정답 확률이 아닙니다.','모델: Xenova/clip-vit-base-patch32'].join('\n');}
$('copy').addEventListener('click',async()=>{if(!lastResult)return;try{await navigator.clipboard.writeText(report());$('actionNotice').textContent='점검 결과를 복사했습니다.';}catch(e){$('actionNotice').textContent='복사 권한을 확인하거나 TXT 저장을 이용하세요.';}});
$('download').addEventListener('click',()=>{if(!lastResult)return;const url=URL.createObjectURL(new Blob(['\ufeff'+report()],{type:'text/plain;charset=utf-8'}));const a=document.createElement('a');a.href=url;a.download='wearcheck-result.txt';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);$('actionNotice').textContent='점검 결과 TXT를 저장했습니다.';});
try{const {pipeline}=await import('https://cdn.jsdelivr.net/npm/@huggingface/transformers@3.7.6');classifier=await pipeline('zero-shot-image-classification','Xenova/clip-vit-base-patch32',{device:'wasm',dtype:'q8'});$('status').textContent='준비 완료. 상품 사진과 카테고리를 선택하세요.';}catch(e){$('status').textContent='모델 로딩 실패: '+e.message;console.error(e);}sync();
</script>
</body></html>
"""
Path("index.html").write_text(html_source, encoding="utf-8")
print("index.html을 생성하였습니다. 브라우저에서 열어 실행하세요.")


index.html을 생성하였습니다. 브라우저에서 열어 실행하세요.
